# Masked 2-unit GRU: three conditions, five seeds each — v8

Three conditions on the **masked** 2-unit GRU (self-connections only, enforced every forward pass), five
seeds each, with per-seed decision regions, averaged trajectories and baseline fixed points, to see how
consistently the rotated, orthogonal solution appears:

1. **baseline** — standard synthetic stimuli (baseline noise 0.1), conflicts trained with random labels.
2. **no_noise** — baseline noise removed from the stimuli, keeping the idle relaxation window. Tests whether
   the noise is what pushes the network to share both tasks across both units.
3. **conflict_trained** — conflict trials trained with audio-wins labels, so masking and conflict training are
   combined.

To support the no-noise condition the trials are generated in-notebook from the v8 specification, with a
`NOISE_STD` knob and a conflict-label switch; everything else is held identical across conditions so only
the one intended thing changes.

## 1. Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import minimize
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
OUT_DIR = Path("./masked_conditions_v8"); OUT_DIR.mkdir(exist_ok=True)
HIDDEN = 2; T = 50; T_ON, T_OFF = 10, 20
CH = {"aud_L":0, "aud_R":1, "vis_L":2, "vis_R":3}
CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]

## 2. Data generator (v8 specification, with a noise knob and a conflict-label switch)

Reproduces the v8 trials: 50 steps, stimulus in steps 10-20, four channels, intensity uniform on [1, 3].
`noise_std=0` gives the no-noise condition while keeping the same trial length and relaxation window.
`conflict_train_label` sets how localisation-conflict training trials are labelled ("random" or "audio_wins");
test conflicts are always labelled stronger-cue-wins for scoring.

In [ ]:
SUBTASKS = ["det_absent","det_auditory_only","det_visual_only",
            "loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
            "loc_multisensory_same_L","loc_multisensory_same_R","det_multisensory",
            "loc_conflict_audL_visR","loc_conflict_audR_visL"]
CONFLICT = ["det_multisensory","loc_conflict_audL_visR","loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
TRAIN_COUNTS = {"det_absent":600,"det_visual_only":600,"det_auditory_only":1200,
                "loc_auditory_only_L":300,"loc_auditory_only_R":300,"loc_visual_only_L":300,"loc_visual_only_R":300,
                "loc_multisensory_same_L":300,"loc_multisensory_same_R":300,
                "loc_conflict_audL_visR":300,"loc_conflict_audR_visL":300}   # det_multisensory is test-only
TEST_PER_TYPE = 100

def make_trial(sub, noise_std, rng, split, conflict_train_label):
    X = rng.normal(0.0, noise_std, (4, T)).astype(np.float32) if noise_std > 0 else np.zeros((4, T), np.float32)
    def add(ch, inten): X[ch, T_ON:T_OFF] += inten
    ai = vi = np.nan
    if sub == "det_absent": lab = 0
    elif sub == "det_auditory_only": i = rng.uniform(1,3); add(0,i); add(1,i); lab = 1
    elif sub == "det_visual_only": i = rng.uniform(1,3); add(2,i); add(3,i); lab = 0
    elif sub == "loc_auditory_only_L": add(0, rng.uniform(1,3)); lab = 3
    elif sub == "loc_auditory_only_R": add(1, rng.uniform(1,3)); lab = 2
    elif sub == "loc_visual_only_L": add(2, rng.uniform(1,3)); lab = 3
    elif sub == "loc_visual_only_R": add(3, rng.uniform(1,3)); lab = 2
    elif sub == "loc_multisensory_same_L": i = rng.uniform(1,3); add(0,i); add(2,i); lab = 3
    elif sub == "loc_multisensory_same_R": i = rng.uniform(1,3); add(1,i); add(3,i); lab = 2
    elif sub == "det_multisensory": i = rng.uniform(1,3); [add(c,i) for c in range(4)]; lab = 1
    elif sub == "loc_conflict_audL_visR":      # audio left, visual right
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(0, ai); add(3, vi)
        lab = (3 if ai > vi else 2) if split == "test" else (3 if conflict_train_label=="audio_wins" else rng.integers(2,4))
    elif sub == "loc_conflict_audR_visL":      # audio right, visual left
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(1, ai); add(2, vi)
        lab = (2 if ai > vi else 3) if split == "test" else (2 if conflict_train_label=="audio_wins" else rng.integers(2,4))
    return X, int(lab), np.float32(ai), np.float32(vi)

def make_split(split, noise_std, conflict_train_label, seed=0):
    rng = np.random.default_rng(1000 + seed + (0 if split=="train" else 7))
    Xs, ys, ts, ais, vis = [], [], [], [], []
    counts = TRAIN_COUNTS if split == "train" else {s: TEST_PER_TYPE for s in SUBTASKS}
    for sub, n in counts.items():
        for _ in range(n):
            X, lab, ai, vi = make_trial(sub, noise_std, rng, split, conflict_train_label)
            Xs.append(X); ys.append(lab); ts.append(sub); ais.append(ai); vis.append(vi)
    return {"X": np.stack(Xs), "y": np.array(ys, np.int64), "types": np.array(ts),
            "aud_int": np.array(ais, np.float32), "vis_int": np.array(vis, np.float32)}

print("Generator ready.")

## 3. Masked GRU (self-connections only) and numpy update

In [ ]:
class MaskedGRU(nn.Module):
    def __init__(self, n_in=4, hidden=2, n_out=4):
        super().__init__(); self.H = hidden; s = 1.0/math.sqrt(hidden)
        self.weight_ih = nn.Parameter(torch.empty(3*hidden, n_in).uniform_(-s, s))
        self.weight_hh = nn.Parameter(torch.empty(3*hidden, hidden).uniform_(-s, s))
        self.bias_ih   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.bias_hh   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.readout   = nn.Linear(hidden, n_out)
        self.register_buffer("mask", torch.eye(hidden).repeat(3, 1))
    def masked_hh(self): return self.weight_hh * self.mask
    def forward(self, x):
        x = x.transpose(1, 2); B, Tt, _ = x.shape; H = self.H; Whh = self.masked_hh()
        Wir, Wiz, Win = self.weight_ih[:H], self.weight_ih[H:2*H], self.weight_ih[2*H:]
        Whr, Whz, Whn = Whh[:H], Whh[H:2*H], Whh[2*H:]
        bir, biz, bin_ = self.bias_ih[:H], self.bias_ih[H:2*H], self.bias_ih[2*H:]
        bhr, bhz, bhn = self.bias_hh[:H], self.bias_hh[H:2*H], self.bias_hh[2*H:]
        h = x.new_zeros(B, H); outs = []
        for t in range(Tt):
            xt = x[:, t, :]
            r = torch.sigmoid(xt @ Wir.T + bir + h @ Whr.T + bhr)
            z = torch.sigmoid(xt @ Wiz.T + biz + h @ Whz.T + bhz)
            n = torch.tanh(xt @ Win.T + bin_ + r * (h @ Whn.T + bhn))
            h = (1 - z) * n + z * h; outs.append(self.readout(h))
        return torch.stack(outs, 1)

def sigmoid(v): return 1.0/(1.0+np.exp(-v))
def masked_params(m):
    Wih = m.weight_ih.detach().numpy(); Whh = m.masked_hh().detach().numpy()
    bih = m.bias_ih.detach().numpy(); bhh = m.bias_hh.detach().numpy(); H = 2; sl = lambda M,i: M[i*H:(i+1)*H]
    return dict(Wir=sl(Wih,0),Wiz=sl(Wih,1),Win=sl(Wih,2),Whr=sl(Whh,0),Whz=sl(Whh,1),Whn=sl(Whh,2),
                bir=bih[:H],biz=bih[H:2*H],bin_=bih[2*H:3*H],bhr=bhh[:H],bhz=bhh[H:2*H],bhn=bhh[2*H:3*H],H=H)
def gru_step(h, x, p):
    r = sigmoid(p["Wir"]@x + p["bir"] + p["Whr"]@h + p["bhr"])
    z = sigmoid(p["Wiz"]@x + p["biz"] + p["Whz"]@h + p["bhz"])
    n = np.tanh(p["Win"]@x + p["bin_"] + r*(p["Whn"]@h + p["bhn"]))
    return (1.0-z)*n + z*h

## 4. Train / evaluate / dynamics helpers

In [ ]:
def train_masked(train, seed, n_epochs=50, lr=1e-3, batch=64):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MaskedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(train["y"])), batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            lo = model(Xb); B, Tt, C = lo.shape
            loss = loss_fn(lo.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model, test):
    model.eval()
    with torch.no_grad(): pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    nc = np.mean([(pred[test["types"]==s]==test["y"][test["types"]==s]).mean() for s in NONCONF])
    return pred, float(nc)

def hidden_traj(test, idx, P):
    out = np.zeros((len(idx), T+1, 2))
    for k, i in enumerate(idx):
        h = np.zeros(2)
        for t in range(T): h = gru_step(h, test["X"][i][:, t], P); out[k, t+1] = h
    return out

def fixed_points(P, inits, tol=1e-10):
    x0 = np.zeros(4)
    def q(h): d = gru_step(h, x0, P) - h; return 0.5*float(d@d)
    found = []
    for h0 in inits:
        r = minimize(q, h0, method="L-BFGS-B", bounds=[(-1.2,1.2)]*2, options=dict(maxiter=400))
        if r.fun < tol: found.append(r.x)
    uniq = []
    for h in found:
        if not any(np.allclose(h, u, atol=1e-3) for u in uniq): uniq.append(h)
    return uniq
def stability(h, P, eps=1e-5):
    x0 = np.zeros(4); f0 = gru_step(h, x0, P); J = np.zeros((2,2))
    for i in range(2):
        hp = h.copy(); hp[i] += eps; J[:, i] = (gru_step(hp, x0, P) - f0)/eps
    m = np.abs(np.linalg.eigvals(J))
    return "stable" if np.all(m<1) else "unstable" if np.all(m>1) else "saddle" 

## 5. Configuration

`CONDITIONS` lists the three experiments. Each entry is (name, noise_std, conflict_train_label).

In [ ]:
CONDITIONS = [
    ("baseline",         0.1, "random"),
    ("no_noise",         0.0, "random"),
    ("conflict_trained", 0.1, "audio_wins"),
]
SEEDS = [0, 1, 2, 3, 4]
N_EPOCHS = 50
GRID = np.linspace(-1.05, 1.05, 300); GX, GY = np.meshgrid(GRID, GRID); _flat = np.stack([GX.ravel(), GY.ravel()], 1)
print("Conditions:", [c[0] for c in CONDITIONS], " seeds:", SEEDS)

## 6. Run all conditions, 5 seeds each, with a per-seed dynamics panel

For each condition the data are generated once (identical across seeds), then five seeds are trained. Each
panel shows the readout decision regions, the averaged trajectory per subtask (endpoints starred by class),
and the baseline fixed points (o stable, X saddle, ^ unstable), so the consistency of the solution across
seeds is visible at a glance.

In [ ]:
summary = {}
for name, noise_std, ctl in CONDITIONS:
    print("\n=== CONDITION:", name, "(noise_std=%.2f, conflict_train_label=%s) ===" % (noise_std, ctl))
    train = make_split("train", noise_std, ctl)
    test  = make_split("test",  noise_std, ctl)
    fig, axes = plt.subplots(1, len(SEEDS), figsize=(4.1*len(SEEDS), 4.3)); axes = np.atleast_1d(axes)
    ncs = []
    for ax, seed in zip(axes, SEEDS):
        model = train_masked(train, seed, N_EPOCHS)
        pred, nc = evaluate(model, test); ncs.append(nc)
        P = masked_params(model); Wro = model.readout.weight.detach().numpy(); bro = model.readout.bias.detach().numpy()
        reg = np.argmax(_flat @ Wro.T + bro, 1).reshape(GX.shape)
        ax.pcolormesh(GX, GY, reg, cmap=ListedColormap(CLASS_COLORS), alpha=0.18, shading="auto", vmin=0, vmax=3)
        ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_aspect("equal")
        ax.set_xlabel("unit 1", fontsize=8); ax.set_ylabel("unit 2", fontsize=8)
        # averaged trajectory per subtask, endpoint starred by predicted class
        for s in SUBTASKS:
            idx = np.where(test["types"] == s)[0]
            if len(idx) == 0: continue
            tr = hidden_traj(test, idx, P).mean(0)
            ax.plot(tr[:, 0], tr[:, 1], color="0.35", lw=1.0, alpha=0.8)
            cls = int(np.argmax(Wro @ tr[-1] + bro))
            ax.scatter(*tr[-1], color=CLASS_COLORS[cls], s=38, marker="*", edgecolor="k", lw=0.4, zorder=6)
        # baseline fixed points
        ends = []
        for i in range(0, len(test["X"]), 11):
            h = np.zeros(2)
            for t in range(T): h = gru_step(h, test["X"][i][:, t], P)
            ends.append(h)
        inits = ends + [np.random.uniform(-1, 1, 2) for _ in range(80)]
        for h in fixed_points(P, inits):
            mk = {"stable":"o","saddle":"X","unstable":"^"}[stability(h, P)]
            ax.scatter(*h, marker=mk, s=95, edgecolor="k", facecolor="white", linewidths=1.5, zorder=7)
        ax.scatter(0, 0, color="k", s=16, zorder=8)
        ax.set_title("seed %d  (nc %.2f)" % (seed, nc), fontsize=10)
    fig.suptitle("Masked 2-unit GRU — %s condition (o stable, X saddle, ^ unstable; stars = trial endpoints by class)" % name, fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(OUT_DIR / ("masked_%s_5seeds.png" % name), dpi=150, bbox_inches="tight"); plt.show()
    summary[name] = ncs
    print("  non-conflict accuracy by seed:", [round(v, 3) for v in ncs])

## 7. Summary

In [ ]:
print("Non-conflict accuracy per condition (5 seeds):")
for name, _, _ in CONDITIONS:
    v = summary[name]; print("  %-18s mean %.3f   seeds %s" % (name, np.mean(v), [round(x,2) for x in v]))
print("\nFigures saved to", OUT_DIR)

## Notes

- All three conditions use the masked model (self-connections only, mask applied every forward pass), so
  the constraint holds in training and testing throughout.
- Only one thing changes between conditions: the stimulus noise (baseline vs no_noise) or the conflict
  training labels (random vs audio_wins). Everything else, including trial structure and the idle
  relaxation window, is identical, so differences in the per-seed dynamics are attributable to that change.
- What to look for: whether the two attractors and the decision-region cross appear rotated relative to the
  unit axes (distributed coding, both units used for both tasks) and how consistently that pattern recurs
  across seeds; and whether removing the noise changes the rotation, which is the hypothesis under test.